In [3]:
from __future__ import annotations

import math
import os
import random
from typing import TYPE_CHECKING, assert_type

import numpy as np
import torch
import torch.nn.functional as F
from shape_extensions import Int, IntVar
from torch import Tensor

B = IntVar("B")
L = IntVar("L")
K = IntVar("K")
N = IntVar("N")
D = IntVar("D")


def seed_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def check_shape(inputs, *shape):
    assert isinstance(inputs, torch.Tensor)
    assert tuple(inputs.shape) == tuple(shape), (tuple(inputs.shape), shape)


def check_probabilities(predicted_probs):
    assert torch.isfinite(predicted_probs).all(), "NaN/inf probability"
    assert ((predicted_probs >= 0) & (predicted_probs <= 1)).all(), "outside [0,1]"


def check_candidates(candidate_mask):
    assert candidate_mask.dtype == torch.bool and candidate_mask.ndim == 3
    assert candidate_mask.any(-1).all(), "empty candidate set"


def check_simplex(predicted_probs, atol=1e-06):
    check_probabilities(predicted_probs)
    torch.testing.assert_close(
        predicted_probs.sum(-1),
        torch.ones_like(predicted_probs.sum(-1)),
        atol=atol,
        rtol=0,
    )


seed_all(1)
torch.set_num_threads(min(4, os.cpu_count() or 1))


# reimplement the original F.one_hot but with Pyrefly compatibility
def one_hot_tokens(
    token_ids: Tensor[[B, L]],
    num_classes: Int[K],
) -> Tensor[[B, L, K]]:
    """Convert token IDs into one-hot vectors."""
    assert token_ids.dtype == torch.long
    assert num_classes > 0

    category_ids = torch.arange(num_classes, device=token_ids.device)
    return (token_ids[..., None] == category_ids).to(torch.long)

# Data
## Encode
$$\mathrm{token\_id}(a)=\operatorname{index}(a,\mathrm{alphabet}).$$

In [4]:
def encode_ascii_bytes(raw: bytes, alphabet: str) -> Tensor:
    lookup = np.full(256, 255, dtype=np.uint8)
    for i, char in enumerate(alphabet):
        lookup[ord(char)] = i

    data = lookup[np.frombuffer(raw, dtype=np.uint8)]
    if (data == 255).any():
        raise ValueError("raw contains characters outside alphabet")

    return torch.from_numpy(data.copy())

In [5]:
def check_encode_text():
    inputs = encode_ascii_bytes(b" abba ", " ab")
    assert inputs.dtype == torch.uint8 and inputs.tolist() == [0, 1, 2, 2, 1, 0]
    assert encode_ascii_bytes(b"", " ab").numel() == 0
    try:
        encode_ascii_bytes(b"?", " ab")
    except ValueError:
        pass
    else:
        raise AssertionError("unknown characters must fail")


check_encode_text()

## Split dataset (text stream) into train, validation, and test sets.
$$x_{\rm train}=x[0:a],\quad x_{\rm valid}=x[a:b],\quad x_{\rm test}=x[b:N].$$

Here `a=train_end`, `b=valid_end`, and `N` is the stream length. These are dataset slices, not an SLM probability formula.

In [6]:
def split_stream(
    ids: Tensor[[N]], train_end: int, valid_end: int
) -> tuple[Tensor, Tensor, Tensor]:
    assert 0 < train_end < valid_end < ids.numel(), (
        "train_end and valid_end must be in [0, N]"
    )
    return ids[:train_end], ids[train_end:valid_end], ids[valid_end:]

In [7]:
def check_split_stream():
    train_tokens, validation_tokens, test_tokens = split_stream(
        torch.arange(100), 90, 95
    )
    assert (train_tokens.numel(), validation_tokens.numel(), test_tokens.numel()) == (
        90,
        5,
        5,
    )
    assert (
        train_tokens[-1] == 89 and validation_tokens[0] == 90 and (test_tokens[0] == 95)
    )
    assert torch.equal(
        torch.cat([train_tokens, validation_tokens, test_tokens]), torch.arange(100)
    )


check_split_stream()

## Gather contiguous training windows
$$X_{b,\ell}=\mathrm{ids}[\mathrm{starts}_b+\ell],\qquad 0\leq\ell<L.$$

`starts` chooses the beginning of each contiguous example. `L=length`. This helper does not predict the next character.

In [8]:
def gather_windows(
    ids: Tensor[[N]], starts: Tensor[[B]], length: Int[L]
) -> Tensor[[B, L]]:
    assert length > 0 and ids.numel() >= length, "length must be in [1, N]"
    assert ((starts >= 0) & (starts <= ids.numel() - length)).all(), (
        "starts must be in [0, N-L]"
    )
    offsets = torch.arange(length, device=ids.device)

    # build matrix: [B, 1] + [1, L] = [B, L]
    #             offsets
    #              0  1  2
    # starts  0   [0, 1, 2]
    #         2   [2, 3, 4]
    #         3   [3, 4, 5]
    indices = starts[:, None] + offsets[None, :]

    return ids[indices].long()

In [9]:
def check_gather_windows():
    got = gather_windows(torch.arange(8, dtype=torch.uint8), torch.tensor([0, 5]), 3)
    assert got.dtype == torch.long
    assert got.tolist() == [[0, 1, 2], [5, 6, 7]]
    check_shape(got, 2, 3)


check_gather_windows()
ALPHABET = " abcdefghijklmnopqrstuvwxyz"
FAKE = encode_ascii_bytes(b" ab " * 128, " ab")
assert len(ALPHABET) == 27

# Forward process $q(x^c_t | x^c_0)$

Some notations:
> * Forward transition $q(x_t | x_{t-1})$: given previous step, how do i get next state
> * Forward marginal $q(x_t | x_0)$: given clean state, what is the dist. at time $t$ (ignoring intermediate steps)
> * Forward posterior $q(x_{t-1} | x_t, x_0)$: given clean state and state at time $t$, what was the likely state at $t-1$

## Write candidate-count schedule

$$n(t)=\exp(t\log K)=K^t,\qquad n(0)=1,\quad n(1)=K.$$

Paper §3.5 uses integer time divided by `T`; we use normalized `time=t` directly.
`n(t)` is the expected number of surviving candidates, not a fixed integer size.

In [10]:
def expected_count(time: Tensor[[B]], vocabulary_size: int) -> Tensor[[B]]:
    assert ((time >= 0) & (time <= 1)).all(), "time must be in [0,1]"
    return torch.exp(time * math.log(vocabulary_size))

In [11]:
def check_expected_count():
    time = torch.tensor([0.0, 0.5, 1.0], dtype=torch.float64)
    update_count = expected_count(time, 16)
    torch.testing.assert_close(
        update_count, torch.tensor([1.0, 4.0, 16.0], dtype=time.dtype)
    )
    assert (update_count[1:] >= update_count[:-1]).all()


check_expected_count()

## Convert a set to its simplex centroid
$$z_{b,\ell,k}=\frac{c_{b,\ell,k}}{\sum_{v=1}^{K}c_{b,\ell,v}}.$$

`c` is Boolean membership; `z` is the simplex centroid fed to the backbone. This normalized representation appears in §3.5 and Appendix 8.2.

In [12]:
def centroid(candidate_mask: Tensor[[B, L, K]]) -> Tensor[[B, L, K]]:
    check_candidates(candidate_mask)
    c_sum = candidate_mask.sum(-1, keepdim=True)

    return candidate_mask.float() / c_sum

In [13]:
def check_centroid():
    candidate_mask = torch.tensor([[[True, False, True], [False, True, False]]])
    centroids = centroid(candidate_mask)
    torch.testing.assert_close(
        centroids, torch.tensor([[[0.5, 0.0, 0.5], [0.0, 1.0, 0.0]]])
    )
    check_simplex(centroids)
    if TYPE_CHECKING:
        example = torch.ones(2, 3, 4, dtype=torch.bool)
        assert_type(centroid(example), Tensor[[2, 3, 4]])


check_centroid()

# Forward candidate Appending (Paper Sec. 3.1)

## Marginal probability (Derive Bernoulli parameter $\phi_{t,k}$)
$$\phi_{t,k} = P(c_{t,k} = 1 | x_0)$$
Because $c_{t,k}$ is binary, we know
$$P(c_{t,k} = 0 | x_0) = 1 - \phi_{t,k}$$
This is a Bernoulli distribution: $$c_{t,k} | x_0 \sim \operatorname{Bern}(\phi_{t,k})$$
Or $$q(c_{t,k} | x_0) = \operatorname{Bern}(c_{t,k}; \phi_{t,k})$$

We want $\phi_t$, so:

The candidate vector is binary:
$$
c_t = (c_{t,1}, c_{t,2}, \ldots, c_{t,K}) \in \{0, 1\}^K
$$
$c_{t,k} = 1$, token is included, else $c_{t,k} = 0$, token is excluded.

SLM chooses the expected candidate-set size
$$
n(t) = K^t
$$
Where $n(0) = K^0 = 1$ and $n(1) = K^1 = K$.

> Equivalently: $\log n(t) = t \log K$, the expected candidate count grows linearly with time.

Summing over all entries tells us how uncertain the state is
$$
| c_t | = \sum_{k=1}^{K} c_{t,k}
$$
being $| c_t | = 1$ means we are certain about the token, else $| c_t | = K$ every token could be possible.

There are $K$ tokens total, $1$ true token and $K - 1$ incorrect tokens
Since $c_t$ is random, the expected number of included tokens is
$$
\begin{align}
\mathbb{E}[|c_t| | x_0] &= \mathbb{E} \big[ \sum_{k=1}^{K} c_{t,k} | x_0 \big] \\
\mathbb{E}[|c_t| | x_0] &= \sum_{k=1}^{K} \mathbb{E} \big[ c_{t,k} | x_0 \big]
\end{align}
$$
Since $c_{t,k}$ is Bernoulli ($\mathbb{E} [c_{t,k}] = P(c_{t,k} = 1 | x_0) = \phi_{t,k}$), we have
$$
\mathbb{E}[|c_t| | x_0] = \sum_{k=1}^{K}  \phi_{t,k} \\
$$
Let their **shared inclusion probability** be $p_t$, then the expected number of included tokens is
$$
\mathbb{E}[|c_t| | x_0] = 1 + (K-1)p_t
$$

To derive $p_t$: We want this expectation to be equal $n(t)$:
$$
1 + (K-1)p_t = n(t)
$$
Solving for $n(t)$,
$$
p_t = \frac{n(t) - 1}{K - 1}
$$

Therefore,
$$
\phi_{t,k} =
\begin{cases}
      1                   , & k=x_0,\\
      \frac{n(t)-1}{K-1}  , & k\neq x_0.
\end{cases}
$$

Now we can write $\phi_t$ as a vector of length $K$: $\phi_t = (\phi_{t,1}, \phi_{t,2}, \ldots, \phi_{t,K})$.
Or using the one-hot vector $e_{x_0}$:
$$
\phi_t = e_{x_0} + (1 - e_{x_0}) \frac{n(t) - 1}{K - 1}
$$


Which is the probability that token $k$ belongs to the candidate set at time $t$:
* For the true token $e_{x_0,k} = 1$:
$$
\sigma_{t,k} = 1 + (1 - 1) p_t = 1
$$
* For the incorrect tokens $e_{x_0,k} = 0$:
$$
\sigma_{t,k} = 0 + (1 - 0) p_t = p_t
$$

In [4]:
def marginal_probs(
    clean_tokens: Tensor[[B, L]], time: Tensor[[B]], vocabulary_size: Int[K]
) -> Tensor[[B, L, K]]:
    assert clean_tokens.dtype == torch.long
    assert ((clean_tokens >= 0) & (clean_tokens < vocabulary_size)).all()

    e0 = one_hot_tokens(clean_tokens, vocabulary_size).to(time.dtype)

    inclusion_prob = (
        (expected_count(time, vocabulary_size) - 1) / (vocabulary_size - 1)
    ).clamp(0, 1)

    return e0 + (1 - e0) * inclusion_prob[:, None, None]

NameError: name 'Tensor' is not defined

In [15]:
def check_marginal_probs():
    inputs = torch.tensor([[1, 2], [0, 1]])
    predicted_probs = marginal_probs(inputs, torch.tensor([0.0, 1.0]), 3)
    assert torch.equal(predicted_probs[0], F.one_hot(inputs[0], 3).float())
    torch.testing.assert_close(predicted_probs[1], torch.ones(2, 3))
    predicted_probs = marginal_probs(torch.tensor([[2]]), torch.tensor([0.5]), 4)
    torch.testing.assert_close(
        predicted_probs, torch.tensor([[[1 / 3, 1 / 3, 1.0, 1 / 3]]])
    )
    check_probabilities(predicted_probs)


check_marginal_probs()

Notation:
* $x^c_t$ corresponds to a complete candidate vector $c_t$
* $c_{t,k}$ corresponds to a complete candidate vector $c_t$

## Forward marginal sampling (Paper Eq. 4)
We can use **inverse-transform sampling** to sample from the Bernoulli distribution. For each candidate $c_{t,k}$, we can draw a uniform random number $U_{b,l,k} \sim \mathcal U[0,1)$ and include the token if $U_{b,l,k} < \phi_{t,b,l,k}$.

$$c_{t,b,\ell,k}=\mathbf1[U_{b,\ell,k}<\phi_{t,b,\ell,k}].$$

This is possible because $c_{t,k}$ are independently sampled from $U_k$, so the probability of obtaining a particular vector is:
$$
P(c_t | x_0) = \prod_k P(c_{t,k} | x_0) = q(c_t | x_0) = \prod_k\operatorname{Bern}(c_{t,k};\phi_{t,k})
$$
$$
q(c_t | x_0) = \operatorname{Bern}(e_{x_0} + (1 - e_{x_0}) \frac{n(t) - 1}{K - 1})
$$

In [16]:
def sample_forward(
    clean_tokens: Tensor[[B, L]],
    time: Tensor[[B]],
    vocabulary_size: Int[K],
    generator=None,
) -> Tensor[[B, L, K]]:
    predicted_probs = marginal_probs(clean_tokens, time, vocabulary_size)

    uniform_samples = torch.rand(
        predicted_probs.shape,
        dtype=predicted_probs.dtype,
        device=predicted_probs.device,
        generator=generator,
    )

    return uniform_samples < predicted_probs

In [17]:
def check_sample_forward():
    inputs = torch.zeros(12000, 1, dtype=torch.long)
    generator = torch.Generator().manual_seed(2026)
    candidate_mask = sample_forward(inputs, torch.full((12000,), 0.5), 4, generator)
    check_candidates(candidate_mask)
    assert candidate_mask[:, :, 0].all()
    assert abs(candidate_mask.float().sum(-1).mean().item() - 2) < 0.04
    first_wrong_included, second_wrong_included = (
        candidate_mask[:, 0, 1].float(),
        candidate_mask[:, 0, 2].float(),
    )
    assert (
        abs((first_wrong_included * second_wrong_included).mean().item() - 1 / 9) < 0.02
    )
    assert sample_forward(inputs[:2], torch.ones(2), 4).all()
    assert sample_forward(inputs[:2], torch.zeros(2), 4).sum(-1).eq(1).all()


check_sample_forward()

## Reverse survival ratio (Paper Eq. 5, 6, 10)

$$\begin{align}
r(s, t) &= \frac{n(s) - 1}{n(t) - 1} = \frac{\exp(s \log K) - 1}{\exp(t \log K) - 1} \\
        &= \frac{\operatorname{expm1}(s \log K)}{\operatorname{expm1}(t \log K)}
\end{align}$$
where $\operatorname{expm1}$ is the numerically stable function $\operatorname{expm1}(x) = \exp(x) - 1$.

This is the probability that a wrong category survives the backward transition, conditional on being present now.

In [18]:
def survival_ratio(
    earlier_time: Tensor[[B]], time: Tensor[[B]], vocabulary_size: int
) -> Tensor[[B]]:
    assert ((0 <= earlier_time) & (earlier_time <= time) & (time <= 1)).all(), (
        "time must be in [0,1] and earlier_time <= time"
    )
    return torch.expm1(earlier_time * math.log(vocabulary_size)) / torch.expm1(
        time * math.log(vocabulary_size)
    )

In [19]:
def check_survival_ratio():
    retention_ratio = survival_ratio(
        torch.tensor([0.0, 0.5], dtype=torch.float64),
        torch.ones(2, dtype=torch.float64),
        4,
    )
    torch.testing.assert_close(
        retention_ratio, torch.tensor([0.0, 1 / 3], dtype=torch.float64)
    )
    retention_ratio = survival_ratio(
        torch.tensor([1e-10], dtype=torch.float64),
        torch.tensor([2e-10], dtype=torch.float64),
        27,
    )
    assert abs(retention_ratio.item() - 0.5) < 1e-08


check_survival_ratio()

## Forward posterior distribution (Paper Eq. 5)
$$
q(x_{t-1}\mid x_t,x_0) = \operatorname{Bern}\left(
x_0+
\left[(1-x_0)\odot x_t\right]
\frac{n(t-1)-1}{n(t)-1}
\right)
$$

In [20]:
def posterior_probs(
    clean_tokens: Tensor[[B, L]],
    candidate_mask: Tensor[[B, L, K]],
    earlier_time: Tensor[[B]],
    time: Tensor[[B]],
) -> Tensor[[B, L, K]]:
    check_candidates(candidate_mask)
    assert candidate_mask.gather(-1, clean_tokens[..., None]).all(), (
        "clean token must be in candidate set"
    )

    clean_one_hot = one_hot_tokens(clean_tokens, candidate_mask.shape[-1]).to(
        time.dtype
    )
    sr = survival_ratio(earlier_time, time, candidate_mask.shape[-1])[:, None, None]

    return clean_one_hot + (1 - clean_one_hot) * candidate_mask * sr

In [21]:
def check_posterior_probs():
    inputs = torch.tensor([[0]])
    candidate_mask = torch.tensor([[[True, True, False, True]]])
    posterior_params = posterior_probs(
        inputs, candidate_mask, torch.tensor([0.5]), torch.tensor([1.0])
    )
    torch.testing.assert_close(
        posterior_params, torch.tensor([[[1.0, 1 / 3, 0.0, 1 / 3]]])
    )
    q0 = posterior_probs(
        inputs, candidate_mask, torch.tensor([0.0]), torch.tensor([0.5])
    )
    torch.testing.assert_close(q0, F.one_hot(inputs, 4).float())


check_posterior_probs()

Notation:
Since $x^c$ is binary, we can count its ones with $\mathbf 1^T$:
$$
\mathbf 1^T x^c = \sum_{k=1}^{K} c_k = \mid x^c \mid
$$
* $\mathbf 1^T x^c > 0$: The set should not be empty
* $\mathbf 1^T x^c_t \geq \mathbf 1^T x^c_{t-1}$: The number of candidates cannot decrease in the forward process
* $ [x_t^c]^T x_{t-1}^c=\mathbf 1^T x_{t-1}^c$: The candidates must remain in the forward process


## Sampling
The forward process starts with one clean token: $x_0^c = e_{x_0}$
and it ends with every token being a candidate $x_T^c = \mathbf 1$
Conceptually:
$$
\{x_0\}=C_0\subseteq C_1\subseteq\cdots\subseteq C_T = \{1,\ldots,K\}.
$$
> Candidates may be added, but never removed
1. The candidate count must not decrease
$$
\mathbf 1^T x_{t-1}^c \leq \mathbf 1^T x_t^c
$$
Or $\mid C_{t-1} \mid \leq \mid C_t \mid$.
2. Every previous candidate must remain
$$
[x_t^c]^T x_{t-1}^c=\mathbf 1^T x_{t-1}^c
$$
We can write $\mid C_t \cap C_{t-1} \mid = \mid C_{t-1} \mid$.

This is also only possible when all early candidates are still present:
$$
C_{t-1}\subseteq C_t
$$

---

The forward transition:
$$
q(x_t^c\mid x_{t-1}^c) = \operatorname{Bern}(\phi)
$$
with (eq. 3)
$$
\phi = x_{t-1}^c + (1-x_{t-1}^c) \frac{n(t)-n(t-1)}{K-n(t-1)}
$$
We have two cases:
1. Token already present $$[x^c_{t-1}]_k = 1$$
then $$\phi_k = 1 + (1-1)a = 1$$
thus $$[x^c_t]_k \sim \operatorname{Bern}(1)$$

2. Token currently absent
If $$[x^c_{t-1}]_k = 0$$
then $$\phi_k = 0 + (1-0)a = a$$
so the token may be added with probability $a$

This direcly implementes candidate appending
$$
C_{t-1}\subseteq C_t
$$

---

The reverse process should instead remove candidates $C_t\supseteq C_{t-1}$. Starting from $C_T=\{1,\ldots,K\}$, remove candidates until only the clean token remain $C_0 = \{x_0\}$

We can calculate the exact reverse posterior as we already have seen:
$$
q(c_s\mid c_t,x_0) = \operatorname{Bern}(\psi)
$$
where
$$
\psi = e_{x_0} + [(1-e_{x_0})\odot c_t]r(s,t)
$$
and
$$
r(s,t) = \frac{n(s)-1}{n(t)-1}
$$

We have three possible cases:
1. clean token
$$
[e_{x_0}]_k = 1 \implies \psi_k = 1 + (1-1)r(s,t) = 1
$$
The clean token must survive
2. An incorrect token still present in $c_t$
$$
[e_{x_0}]_k = 0 \text{ and } [c_t]_k = 1 \implies \psi_k = 0 + (1-0)r(s,t) = r(s,t)
$$
It survives backward to the earlier time with probability $r(s, t)$.
A token absent from $c_t$
$$
c{t,k} = 0 \implies \psi_k = 0 + (1-0)0r(s,t) = 0
$$
A token already absent from $C_t$ cannot reappear in the smaller earlier set $C_s$. This guarantees $C_s \subseteq C_t$.

---

During generation $x_0$ is unknown. The network predicts a categorical distribution over the possible clean token
$$
[\operatorname{NN}_\theta(x_t^c,t)]^T (\mathbf1-x_t^c) = 0
$$
Using $p_\theta = \operatorname{NN}_\theta(x_t^c,t)$ and expanding the dot product:
$$
\sum_k p_{\theta,k}(1-[x_t^c]_k)=0.
$$
We have two cases:
1. The candidate is 1 $[x_t^c]_k = 1$, the token contributes nothing to the sum
2. The candidate is 0 $[x_t^c]_k = 0$, the token contributes $p_{\theta,k}$ to the sum

Since the sum must be zero, the network must assign zero probability to all tokens not present in the candidate set. This guarantees that the network will never predict a token that is not present in the candidate set.

## Restrict prediction to candidate set (Paper Seq. 3.2)
The network return unrestricted logits:
$$
\ell=(\ell_1,\ldots,\ell_K)
$$
The softmax will be then:
$$
p_{\theta,k} = \frac{\exp(\ell_k)} {\sum_v\exp(\ell_v)}
$$
which assigns non-zero probability to all tokens. We want to restrict the network to only predict candidates:
We can set
$$
\widetilde\ell_k = \begin{cases} \ell_k,&c_{t,k}=1,\\ -\infty,&c_{t,k}=0. \end{cases}
$$
exponentiating gives:
$$
\exp(\widetilde\ell_k) = \begin{cases} \exp(\ell_k),&c_{t,k}=1,\\ 0,&c_{t,k}=0. \end{cases}
$$
since $c_{t,k}$ is binary we can write the condition as:
$$
\exp(\widetilde\ell_k) = c_{t,k} \exp(\ell_k)
$$
Now the softmax looks like
$$
p_{\theta,k} = \frac{c_{t,k}\exp(\ell_k)} {\sum_vc_{t,v}\exp(\ell_v)}
$$

Finally, the cross-entropy loss needs the $\log p_{\theta, k}$, so we take the logarithm of it

$$
\begin{aligned}
\log p_{\theta,k} &= \log\exp(\widetilde\ell_k) - \log\sum_v\exp(\widetilde\ell_v) \\
            &= \widetilde\ell_k - \operatorname{logsumexp}_v(\widetilde\ell_v)
\end{aligned}
$$

In [26]:
def masked_log_probs(
    logits: Tensor[[B, L, K]], candidate_mask: Tensor[[B, L, K]]
) -> Tensor[[B, L, K]]:
    check_candidates(candidate_mask)
    assert logits.shape == candidate_mask.shape
    mask = logits.masked_fill(~candidate_mask, float("-inf"))
    return F.log_softmax(mask, dim=-1)

In [27]:
def check_masked_log_probs():
    centroids = torch.tensor([[[0.0, 999.0, math.log(3.0)]]], requires_grad=True)
    candidate_mask = torch.tensor([[[True, False, True]]])
    log_probabilities = masked_log_probs(centroids, candidate_mask)
    torch.testing.assert_close(
        log_probabilities.exp(), torch.tensor([[[0.25, 0.0, 0.75]]])
    )
    assert torch.isneginf(log_probabilities[0, 0, 1])
    (-log_probabilities[0, 0, 0]).backward()
    assert torch.isfinite(centroids.grad).all() and centroids.grad[0, 0, 1] == 0


check_masked_log_probs()

# Reverse candidate pruning (Paper Seq. 3.2)

## Parameterized reverse transition (Paper Eq. 6)
The posterior can be formulate using a neural network $\theta$ like:
$$
p_\theta(x_{t-1}^c\mid x_t^c) = \operatorname{Bern}\left( \left[ \operatorname{NN}_\theta(x_t^c,t) + \left(1-\operatorname{NN}_\theta(x_t^c,t)\right) \frac{n(t-1)-1}{n(t)-1} \right] \odot x_t^c \right)
$$

In [28]:
def reverse_probs(
    clean_probabilities: Tensor[[B, L, K]],
    candidate_mask: Tensor[[B, L, K]],
    earlier_time: Tensor[[B]],
    time: Tensor[[B]],
) -> Tensor[[B, L, K]]:
    check_candidates(candidate_mask)
    check_simplex(clean_probabilities)
    assert (
        clean_probabilities.shape == candidate_mask.shape
        and (clean_probabilities[~candidate_mask] == 0).all()
    ), "clean_probabilities must be zero outside candidate_mask"

    survival = survival_ratio(earlier_time, time, candidate_mask.shape[-1])[
        :, None, None
    ]
    return candidate_mask.to(clean_probabilities.dtype) * (
        clean_probabilities + (1 - clean_probabilities) * survival
    )

In [29]:
def check_reverse_probs():
    predicted_probs = torch.tensor([[[0.25, 0.75, 0.0, 0.0]]])
    candidate_mask = torch.tensor([[[True, True, False, False]]])
    output = reverse_probs(
        predicted_probs, candidate_mask, torch.tensor([0.5]), torch.tensor([1.0])
    )
    torch.testing.assert_close(output, torch.tensor([[[0.5, 5 / 6, 0.0, 0.0]]]))
    torch.testing.assert_close(
        reverse_probs(
            predicted_probs, candidate_mask, torch.tensor([0.0]), torch.tensor([1.0])
        ),
        predicted_probs,
    )
    inputs = torch.tensor([[1]])
    oracle = F.one_hot(inputs, 4).float()
    torch.testing.assert_close(
        reverse_probs(oracle, candidate_mask, torch.tensor([0.5]), torch.tensor([1.0])),
        posterior_probs(
            inputs, candidate_mask, torch.tensor([0.5]), torch.tensor([1.0])
        ),
    )


check_reverse_probs()

# Training Procedure (Paper Seq. 3.3)

## Bernoulli KL (Paper Eq. 8)
The KL divergence between two multivariate Bernoulli distributions is:
> https://math.stackexchange.com/questions/2604566/kl-divergence-between-two-multivariate-bernoulli-distribution

$$
D_{\mathrm{KL}} \left[ \operatorname{Bern}(q) \| \operatorname{Bern}(p) \right] = \sum_{k:c_{t,k}=1} \left[ q_k\log\frac{q_k}{p_k} + (1-q_k)\log\frac{1-q_k}{1-p_k} \right]
$$
The paper uses:
* $q_k = \text{gd}^k(c_t)$: exact posterior parameter (`posterior_params`)
* $p_k = \text{pred}_\theta^k(c_t)$: learned reverse parameter (`predicted_probs`)

Using the log identity $\log(a/b) = \log a - \log b$ we can write the KL (for one token) as:
$$
d(q, p) = q \log q - q \log p + (1 - q) \log (1 - q) - (1 - q) \log (1 - p)
$$

In [ ]:
def bernoulli_kl(
    target_probs: Tensor[[B, L, K]], predicted_probs: Tensor[[B, L, K]]
) -> Tensor[[B, L, K]]:
    assert target_probs.shape == predicted_probs.shape, (
        "target_probs and predicted_probs must have the same shape"
    )
    check_probabilities(target_probs)
    check_probabilities(predicted_probs)
    target_probs, predicted_probs = (target_probs.double(), predicted_probs.double())
    return (
        torch.special.xlogy(target_probs, target_probs)
        - torch.special.xlogy(target_probs, predicted_probs)
        + torch.special.xlogy(1 - target_probs, 1 - target_probs)
        - torch.special.xlogy(1 - target_probs, 1 - predicted_probs)
    )

In [ ]:
def check_bernoulli_kl():
    posterior_params = torch.tensor([[[0.0, 1.0, 0.5]]], dtype=torch.float64)
    torch.testing.assert_close(
        bernoulli_kl(posterior_params, posterior_params),
        torch.zeros_like(posterior_params),
    )
    predicted_probs = torch.tensor([[[0.2, 0.8, 0.25]]], dtype=torch.float64)
    expected = torch.tensor(
        [[[-math.log(0.8), -math.log(0.8), 0.5 * math.log(4 / 3)]]], dtype=torch.float64
    )
    torch.testing.assert_close(
        bernoulli_kl(posterior_params, predicted_probs), expected
    )
    assert torch.isinf(bernoulli_kl(torch.ones(1, 1, 1), torch.zeros(1, 1, 1))).all()


check_bernoulli_kl()

Notation: The probability mass function of a Bernoulli:
$$
\operatorname{Bern}(x;p) = p^x (1 - p)^{1-p}
$$
This describes that sampling $1$ with probability $p$ and samping $0$ with probability $1-p$.

For a multivariate Bernoulli, the probability of a vector $\mathbf x$ is the product of the probabilities of each entry:

$$
\operatorname{Bern}(\mathbf x;\boldsymbol\psi) = \prod_{k=1}^K P(X_k = x_k) = \prod_{k=1}^K \psi_k^{x_k}(1-\psi_k)^{1-x_k}.
$$

## Reconstruction term (Paper section 3.3)
$$
L_0 = -\mathbb E_{q(x_1^c\mid x_0^c)} \left[ \log p_\theta(x_0^c\mid x_1^c) \right].
$$
We set $t = 1$, so $t-1 = 0$ to the learned reverse transition $p_\theta(x_0^c\mid x_1^c)$.

The reverse survival ration become $0$, therefore
$$
p_\theta(x_0^c\mid x_1^c) = \operatorname{Bern}(x_0^c; \operatorname{NN}_\theta(x_1^c, 1) \odot x_1^c)
$$

We can now expand the conditional bernoulli probability:
$$
p_\theta(x_0^c\mid x_1^c) = \prod_k \widehat\psi_k^{\,x_{0,k}^c} (1-\widehat\psi_k)^{1-x_{0,k}^c}.
$$
Take the logarithm
$$
\log p_\theta(x_0^c\mid x_1^c) = \sum_k \left[ x_{0,k}^c\log\widehat\psi_k + (1-x_{0,k}^c)\log(1-\widehat\psi_k) \right]
$$
We rewrite this sum as a inner product:
1. True-token term (first term):
Becasue $x_0^c$ is one-hot, only one component is $1$. Suppose the true token is $j$:
$$
\left\langle \widehat{\boldsymbol\psi},x_0^c \right\rangle = \sum_k\widehat\psi_kx_{0,k}^c = \widehat\psi_j.
$$

Thus,
$$
\sum_kx_{0,k}^c\log\widehat\psi_k = \log \left\langle \widehat{\boldsymbol\psi},x_0^c \right\rangle
$$

2. Wrong-token term (second term):
The factor $1 - x_{0,k}^c$ is $1$ for all tokens except the true token.

Tokens outside the candidate set also get $0$, because we have:
$$
x_{1,k}^c=0 \quad\Longrightarrow\quad \widehat\psi_k=0
$$
thus, $\log(1-\widehat\psi_k) = 0$

Since the true token $x^c_0$ always belongs to the candidate set $x^c_0 \subseteq x^c_1$, consider $x_{1,k}^c-x_{0,k}^c$:

| Token type | $x_{1,k}^c$ | $x_{0,k}^c$ | $x_{1,k}^c-x_{0,k}^c$ |
|---|---:|---:|---:|
| True token | 1 | 1 | 0 |
| Wrong current candidate | 1 | 0 | 1 |
| Outside candidate set | 0 | 0 | 0 |

$$
\sum_k (1-x_{0,k}^c)\log(1-\widehat\psi_k) = \sum_k (x_{1,k}^c-x_{0,k}^c)\log(1-\widehat\psi_k)
$$

We can now rewrite this as an inner product:
$$
\sum_k (x_{1,k}^c-x_{0,k}^c) \log(1-\widehat\psi_k) = \left\langle x_1^c-x_0^c, \log(1-\widehat{\boldsymbol\psi}) \right\rangle
$$

Combining both rewrites, we have:
$$
\log p_\theta(x_0^c\mid x_1^c) = \log \left\langle \widehat{\boldsymbol\psi},x_0^c \right\rangle + \left\langle x_1^c-x_0^c, \log(1-\widehat{\boldsymbol\psi}) \right\rangle
$$

The first term is the probability that the true token survives and the second term is the probability that every wrong current candidate disappears

In [ ]:
def reconstruction_nll(
    clean_tokens: Tensor[[B, L]], predicted_probs: Tensor[[B, L, K]]
) -> Tensor[[B, L]]:
    check_probabilities(predicted_probs)
    predcited_probs = predicted_probs.double()
    target_one_hot = one_hot_tokens(clean_tokens, predicted_probs.shape[-1]).double()
    return -(
        torch.special.xlogy(target_one_hot, predicted_probs)
        + torch.special.xlogy(1 - target_one_hot, 1 - predicted_probs)
    ).sum(-1)

In [ ]:
def check_reconstruction_nll():
    predicted_probs = torch.tensor([[[0.25, 0.75, 0.0]]], dtype=torch.float64)
    loss = reconstruction_nll(torch.tensor([[0]]), predicted_probs)
    torch.testing.assert_close(
        loss, torch.tensor([[-2 * math.log(0.25)]], dtype=torch.float64)
    )
    assert (
        reconstruction_nll(
            torch.tensor([[1]]), torch.tensor([[[0.0, 1.0, 0.0]]])
        ).item()
        == 0
    )


check_reconstruction_nll()